# CVPR_CH4CHL - Data Review (EDA)

**Platform**: Colab (Google Drive mount)
**Runtime**: CPU
**Estimated time**: ~10 min

## Goals
1. Understand the 133 COCO-WholeBody keypoint structure
2. Patient / video / frame distribution statistics
3. Analyse keypoint confidence scores and decide a filtering policy
4. Track 1 EVGS item distribution + Track 2 gait subtype distribution
5. Visualise keypoint sequences
6. Gait-cycle detection prototype
7. Joint-angle computation prototype

In [ ]:
# === Cell 1: Setup & Imports ===
import json
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from collections import Counter, defaultdict
from pathlib import Path
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
print('Imports OK')

In [ ]:
# === Cell 2: Mount Drive & Config ===
import time

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # === Google Drive paths (persistent storage) ===
    DRIVE_ROOT = '/content/drive/MyDrive/CVPR_CH4CHL'
    DRIVE_DATA = f'{DRIVE_ROOT}/1_data/raw'

    # === Local disk paths (fast, ephemeral) ===
    LOCAL_DATA = '/content/data'
    os.makedirs(LOCAL_DATA, exist_ok=True)

    # --- Step 1: Copy labels to local ---
    for f in ['track1_train.json', 'track2_train.json']:
        src = f'{DRIVE_DATA}/{f}'
        dst = f'{LOCAL_DATA}/{f}'
        if not os.path.exists(dst):
            print(f'Copying {f} to local...')
            os.system(f'cp "{src}" "{dst}"')

    # --- Step 2: Extract dataset.tar.bz2 to LOCAL disk (not Drive!) ---
    DATASET_DIR = f'{LOCAL_DATA}/dataset'
    TAR_FILE = f'{DRIVE_DATA}/dataset.tar.bz2'

    if not os.path.exists(DATASET_DIR):
        assert os.path.exists(TAR_FILE), (
            f'dataset.tar.bz2 not found at {TAR_FILE}\n'
            f'Please upload it to Google Drive: MyDrive/CVPR_CH4CHL/1_data/raw/dataset.tar.bz2'
        )
        print(f'Extracting dataset.tar.bz2 to local disk ({LOCAL_DATA})...')
        print('(This takes ~2-5 min for 339K small files, but only once per session)')
        t0 = time.time()
        os.system(f'tar -xjf "{TAR_FILE}" -C "{LOCAL_DATA}"')
        print(f'Done in {time.time()-t0:.0f}s')

        # Handle case where tar extracts into a subfolder
        if not os.path.exists(DATASET_DIR):
            extracted = [d for d in os.listdir(LOCAL_DATA)
                         if os.path.isdir(f'{LOCAL_DATA}/{d}') and d != 'dataset']
            if extracted:
                os.rename(f'{LOCAL_DATA}/{extracted[0]}', DATASET_DIR)
    else:
        print(f'Dataset already extracted at {DATASET_DIR}')

    TRACK1_JSON = f'{LOCAL_DATA}/track1_train.json'
    TRACK2_JSON = f'{LOCAL_DATA}/track2_train.json'

else:
    # === Local fallback ===
    DRIVE_ROOT = os.environ.get('CV4CHL_REPO_ROOT', os.getcwd())
    DATA_DIR = f'{DRIVE_ROOT}/1_data/raw'
    DATASET_DIR = f'{DATA_DIR}/dataset'
    TRACK1_JSON = f'{DATA_DIR}/track1_train.json'
    TRACK2_JSON = f'{DATA_DIR}/track2_train.json'

# Verify
for p in [TRACK1_JSON, TRACK2_JSON, DATASET_DIR]:
    assert os.path.exists(p), f'Missing: {p}'

n_patients = len([d for d in os.listdir(DATASET_DIR) if d.isdigit()])
print(f'\nPlatform: {"Colab" if IN_COLAB else "Local"}')
print(f'Dataset: {DATASET_DIR} ({n_patients} patients)')
print(f'Labels: {TRACK1_JSON}, {TRACK2_JSON}')

In [ ]:
# === Cell 3: Load Labels ===
with open(TRACK1_JSON) as f:
    track1_labels = json.load(f)
with open(TRACK2_JSON) as f:
    track2_labels = json.load(f)

print(f'Track 1 train: {len(track1_labels)} patients')
print(f'Track 2 train: {len(track2_labels)} patients')

# Track 1 test & Track 2 test IDs (from competition description)
TRACK1_TEST_IDS = [4, 5, 18, 26, 28, 40, 42, 43, 47, 48, 53, 54, 72, 78, 83, 85]
TRACK2_TEST_IDS = [4, 6, 7, 13, 26, 35, 39, 42, 50]

track1_train_ids = sorted([p['patient_id'] for p in track1_labels])
track2_train_ids = sorted([p['patient_id'] for p in track2_labels])

# Check overlap between tracks
overlap = set(track1_train_ids) & set(track2_train_ids)
print(f'\nTrack 1 train IDs: {track1_train_ids}')
print(f'Track 2 train IDs: {track2_train_ids}')
print(f'Overlap (in both tracks): {sorted(overlap)} ({len(overlap)} patients)')

# All patient IDs in dataset
all_dataset_ids = sorted([int(d) for d in os.listdir(DATASET_DIR) if d.isdigit()])
print(f'\nAll patient folders in dataset: {len(all_dataset_ids)}')

## 1. COCO-WholeBody 133 keypoint structure

Sapiens-2B emits 133 keypoints in COCO-WholeBody format:
- **Body (0-16)**: 17 main joints — nose, eyes, ears, shoulders, elbows, wrists, hips, knees, ankles
- **Feet (17-22)**: 6 foot points — left/right big toe, small toe, heel
- **Face (23-90)**: 68 facial landmarks
- **Left hand (91-111)**: 21 left-hand points
- **Right hand (112-132)**: 21 right-hand points

**Gait analysis only needs body + feet (23 keypoints)**; face/hands are noise for this task.

In [ ]:
# === Cell 4: Keypoint Definition ===
# COCO-WholeBody 133 keypoint names
BODY_KP_NAMES = [
    'nose',               # 0
    'left_eye',           # 1
    'right_eye',          # 2
    'left_ear',           # 3
    'right_ear',          # 4
    'left_shoulder',      # 5
    'right_shoulder',     # 6
    'left_elbow',         # 7
    'right_elbow',        # 8
    'left_wrist',         # 9
    'right_wrist',        # 10
    'left_hip',           # 11
    'right_hip',          # 12
    'left_knee',          # 13
    'right_knee',         # 14
    'left_ankle',         # 15
    'right_ankle',        # 16
]

FEET_KP_NAMES = [
    'left_big_toe',       # 17
    'left_small_toe',     # 18
    'left_heel',          # 19
    'right_big_toe',      # 20
    'right_small_toe',    # 21
    'right_heel',         # 22
]

# Groups for analysis
BODY_IDX = list(range(0, 17))    # 17 body joints
FEET_IDX = list(range(17, 23))   # 6 feet points
FACE_IDX = list(range(23, 91))   # 68 face points
LHAND_IDX = list(range(91, 112)) # 21 left hand points
RHAND_IDX = list(range(112, 133))# 21 right hand points

# Gait-relevant keypoints (body + feet = 23)
GAIT_IDX = BODY_IDX + FEET_IDX
GAIT_KP_NAMES = BODY_KP_NAMES + FEET_KP_NAMES

# Skeleton connections for visualization (body + feet)
SKELETON_EDGES = [
    # Head
    (0, 1), (0, 2), (1, 3), (2, 4),
    # Torso
    (5, 6), (5, 11), (6, 12), (11, 12),
    # Left arm
    (5, 7), (7, 9),
    # Right arm
    (6, 8), (8, 10),
    # Left leg
    (11, 13), (13, 15),
    # Right leg
    (12, 14), (14, 16),
    # Left foot
    (15, 17), (15, 18), (15, 19), (17, 18),
    # Right foot
    (16, 20), (16, 21), (16, 22), (20, 21),
]

print(f'Body: {len(BODY_IDX)} keypoints')
print(f'Feet: {len(FEET_IDX)} keypoints')
print(f'Face: {len(FACE_IDX)} keypoints')
print(f'Hands: {len(LHAND_IDX) + len(RHAND_IDX)} keypoints')
print(f'Gait-relevant: {len(GAIT_IDX)} keypoints')
print(f'Total: 133')

## 2. Patient / video / frame statistics

In [ ]:
# === Cell 5: Scan Dataset — Patient / Video / Frame Statistics ===
patient_stats = []

for pid_dir in sorted(os.listdir(DATASET_DIR)):
    pid_path = os.path.join(DATASET_DIR, pid_dir)
    if not os.path.isdir(pid_path) or not pid_dir.isdigit():
        continue
    pid = int(pid_dir)

    videos = sorted([v for v in os.listdir(pid_path)
                     if os.path.isdir(os.path.join(pid_path, v))])

    for video_name in videos:
        video_path = os.path.join(pid_path, video_name)
        frames = [f for f in os.listdir(video_path) if f.endswith('.json')]
        n_frames = len(frames)

        # Parse video name: {pid}-{session}_{view}_{start-end}_filtered_pose
        parts = video_name.split('_')
        view = parts[1] if len(parts) >= 2 else 'unknown'
        session = parts[0].split('-')[1] if '-' in parts[0] else 'unknown'

        patient_stats.append({
            'patient_id': pid,
            'session': session,
            'view': view,
            'video_name': video_name,
            'n_frames': n_frames,
            'video_path': video_path,
        })

df_stats = pd.DataFrame(patient_stats)
print(f'Total videos: {len(df_stats)}')
print(f'Total patients: {df_stats["patient_id"].nunique()}')
print(f'Total frames: {df_stats["n_frames"].sum():,}')
print(f'\nFrames per video:')
print(df_stats['n_frames'].describe().to_string())
print(f'\nViews distribution:')
print(df_stats['view'].value_counts().to_string())
print(f'\nVideos per patient:')
print(df_stats.groupby('patient_id').size().describe().to_string())

In [ ]:
# === Cell 6: Visualize Patient / Video / Frame Distributions ===
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 6a: Frames per video histogram
axes[0].hist(df_stats['n_frames'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Frames per video')
axes[0].set_ylabel('Count')
axes[0].set_title('Frame Count Distribution')
axes[0].axvline(df_stats['n_frames'].median(), color='red', ls='--', label=f"median={df_stats['n_frames'].median():.0f}")
axes[0].legend()

# 6b: Videos per patient
vids_per_patient = df_stats.groupby('patient_id').size()
axes[1].hist(vids_per_patient, bins=range(1, vids_per_patient.max()+2), color='coral', edgecolor='white', align='left')
axes[1].set_xlabel('Videos per patient')
axes[1].set_ylabel('Count')
axes[1].set_title('Videos per Patient')

# 6c: View coverage per patient (heatmap-like)
view_pivot = df_stats.groupby(['patient_id', 'view']).size().unstack(fill_value=0)
view_coverage = (view_pivot > 0).sum(axis=1)
axes[2].hist(view_coverage, bins=range(1, 6), color='seagreen', edgecolor='white', align='left')
axes[2].set_xlabel('Number of distinct views')
axes[2].set_ylabel('Patients')
axes[2].set_title('View Coverage per Patient')

plt.tight_layout()
plt.show()

# Show patients with missing views
full_views = {'forward', 'backward', 'left', 'right'}
missing_views = []
for pid, group in df_stats.groupby('patient_id'):
    views = set(group['view'].unique())
    missing = full_views - views
    if missing:
        missing_views.append({'patient_id': pid, 'missing_views': sorted(missing), 'available': sorted(views)})
if missing_views:
    print(f'\nPatients with incomplete views ({len(missing_views)}/{df_stats["patient_id"].nunique()}):')
    for m in missing_views[:10]:
        print(f'  Patient {m["patient_id"]}: missing {m["missing_views"]}, has {m["available"]}')
    if len(missing_views) > 10:
        print(f'  ... and {len(missing_views) - 10} more')
else:
    print('\nAll patients have all 4 views.')

## 3. Keypoint confidence-score analysis

Goal: confirm that body/feet keypoints carry substantially higher confidence than face/hands so a downstream filtering policy can be justified.

In [ ]:
# === Cell 7: Keypoint Confidence Score Analysis ===
# Sample frames from diverse patients and views
np.random.seed(42)
sample_videos = df_stats.sample(min(30, len(df_stats)), random_state=42)

all_scores = []  # shape will be (n_samples, 133)

for _, row in sample_videos.iterrows():
    video_path = row['video_path']
    frame_files = sorted([f for f in os.listdir(video_path) if f.endswith('.json')])
    # Sample 5 frames per video
    sample_indices = np.linspace(0, len(frame_files)-1, min(5, len(frame_files)), dtype=int)
    for idx in sample_indices:
        fpath = os.path.join(video_path, frame_files[idx])
        with open(fpath) as f:
            data = json.load(f)
        if data['instance_info']:
            scores = data['instance_info'][0]['keypoint_scores']
            all_scores.append(scores)

all_scores = np.array(all_scores)
print(f'Sampled {len(all_scores)} frames from {len(sample_videos)} videos')

# Compute per-group statistics
groups = {
    'Body (0-16)': BODY_IDX,
    'Feet (17-22)': FEET_IDX,
    'Face (23-90)': FACE_IDX,
    'L.Hand (91-111)': LHAND_IDX,
    'R.Hand (112-132)': RHAND_IDX,
}

print(f'\nConfidence Score by Group:')
print(f'{"Group":<20} {"Mean":>8} {"Std":>8} {"Min":>8} {"Q25":>8} {"Median":>8} {"Q75":>8} {"Max":>8}')
print('-' * 88)
for name, indices in groups.items():
    g = all_scores[:, indices].flatten()
    print(f'{name:<20} {g.mean():8.3f} {g.std():8.3f} {g.min():8.3f} '
          f'{np.percentile(g,25):8.3f} {np.median(g):8.3f} {np.percentile(g,75):8.3f} {g.max():8.3f}')

# Boxplot
fig, ax = plt.subplots(figsize=(10, 4))
group_data = [all_scores[:, idx].flatten() for idx in groups.values()]
bp = ax.boxplot(group_data, labels=groups.keys(), patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
ax.set_ylabel('Confidence Score')
ax.set_title('Keypoint Confidence by Body Group')
ax.axhline(0.5, color='red', ls='--', alpha=0.5, label='Threshold=0.5')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# === Cell 8: Per-Keypoint Confidence Heatmap (Body + Feet only) ===
# Mean confidence per gait-relevant keypoint
gait_scores = all_scores[:, GAIT_IDX]  # (n_samples, 23)
mean_scores = gait_scores.mean(axis=0)
std_scores = gait_scores.std(axis=0)

fig, ax = plt.subplots(figsize=(14, 3))
colors = ['steelblue' if s > 0.8 else 'orange' if s > 0.5 else 'red' for s in mean_scores]
bars = ax.bar(range(len(GAIT_KP_NAMES)), mean_scores, yerr=std_scores,
              color=colors, edgecolor='white', capsize=2)
ax.set_xticks(range(len(GAIT_KP_NAMES)))
ax.set_xticklabels(GAIT_KP_NAMES, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Mean Confidence')
ax.set_title('Per-Keypoint Confidence (Body + Feet)')
ax.axhline(0.8, color='gray', ls=':', alpha=0.5)
ax.set_ylim(0, 1.1)

# Legend
legend_elements = [
    mpatches.Patch(facecolor='steelblue', label='>0.8 (reliable)'),
    mpatches.Patch(facecolor='orange', label='0.5-0.8 (caution)'),
    mpatches.Patch(facecolor='red', label='<0.5 (unreliable)'),
]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.show()

## 4. Track 1 — EVGS Item Distribution

In [ ]:
# === Cell 9: Track 1 — EVGS Item Distribution Heatmap ===
EVGS_ITEM_NAMES = [
    '1:Initial Contact',
    '2:Heel Rise',
    '3:Max Ankle Dorsi (Stance)',
    '4:Hindfoot Valgus/Varus',
    '5:Foot Rotation',
    '6:Foot Clearance (Swing)',
    '7:Max Ankle Dorsi (Swing)',
    '8:Knee Ext Terminal Swing',
    '9:Peak Knee Ext (Stance)',
    '10:Peak Knee Flex (Swing)',
    '11:Peak Hip Ext (Stance)',
    '12:Peak Hip Flex (Swing)',
    '13:Pelvic Rotation',
    '14:Pelvic Obliquity',
    '15:Lateral Trunk Shift',
    '16:Peak Sagittal Trunk',
    '17:Pelvic Tilt',
]

# View assignment per EVGS item (sagittal vs coronal)
EVGS_VIEW = {
    1: 'sagittal', 2: 'sagittal', 3: 'sagittal',
    4: 'coronal', 5: 'coronal',
    6: 'sagittal', 7: 'sagittal',
    8: 'sagittal', 9: 'sagittal', 10: 'sagittal',
    11: 'sagittal', 12: 'sagittal',
    13: 'coronal', 14: 'coronal', 15: 'coronal',
    16: 'sagittal', 17: 'sagittal',
}

# Build positive rate matrix
pos_rates = np.zeros((17, 2))  # (items, L/R)
for i, item_num in enumerate(range(1, 18)):
    left_vals = [p['left'][str(item_num)] for p in track1_labels]
    right_vals = [p['right'][str(item_num)] for p in track1_labels]
    pos_rates[i, 0] = np.mean(left_vals)
    pos_rates[i, 1] = np.mean(right_vals)

fig, ax = plt.subplots(figsize=(6, 10))
im = ax.imshow(pos_rates, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Left', 'Right'])
ax.set_yticks(range(17))
ax.set_yticklabels(EVGS_ITEM_NAMES, fontsize=8)

# Annotate cells with values and view type
for i in range(17):
    view_tag = 'S' if EVGS_VIEW[i+1] == 'sagittal' else 'C'
    for j in range(2):
        val = pos_rates[i, j]
        color = 'white' if val > 0.5 else 'black'
        ax.text(j, i, f'{val:.0%}\n({view_tag})', ha='center', va='center',
                fontsize=7, color=color)

ax.set_title('EVGS Positive Rate (S=sagittal, C=coronal)')
plt.colorbar(im, ax=ax, shrink=0.6, label='Positive Rate')
plt.tight_layout()
plt.show()

# Flag imbalanced items
print('Severely imbalanced items (pos < 15%):')
for i in range(17):
    for side_idx, side in enumerate(['Left', 'Right']):
        if pos_rates[i, side_idx] < 0.15:
            print(f'  Item {i+1} ({side}): {pos_rates[i, side_idx]:.1%}')

In [ ]:
# === Cell 10: Track 1 — Total Score Distribution & L/R Correlation ===
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 10a: Total score distribution
left_totals = [p['left']['Total'] for p in track1_labels]
right_totals = [p['right']['Total'] for p in track1_labels]
axes[0].hist(left_totals, bins=range(0, 20), alpha=0.6, label='Left', color='steelblue', edgecolor='white')
axes[0].hist(right_totals, bins=range(0, 20), alpha=0.6, label='Right', color='coral', edgecolor='white')
axes[0].set_xlabel('Total EVGS Score (sum of 17 items)')
axes[0].set_ylabel('Count')
axes[0].set_title('Total Score Distribution')
axes[0].legend()

# 10b: Left vs Right total score scatter
axes[1].scatter(left_totals, right_totals, alpha=0.5, s=30)
axes[1].plot([0, 17], [0, 17], 'r--', alpha=0.3)
axes[1].set_xlabel('Left Total')
axes[1].set_ylabel('Right Total')
axes[1].set_title(f'L vs R Total (corr={np.corrcoef(left_totals, right_totals)[0,1]:.2f})')

# 10c: Per-item L/R agreement rate
agreement = []
for item_num in range(1, 18):
    left_vals = [p['left'][str(item_num)] for p in track1_labels]
    right_vals = [p['right'][str(item_num)] for p in track1_labels]
    agree = sum(l == r for l, r in zip(left_vals, right_vals)) / len(left_vals)
    agreement.append(agree)

axes[2].barh(range(17), agreement, color='seagreen', edgecolor='white')
axes[2].set_yticks(range(17))
axes[2].set_yticklabels([f'Item {i+1}' for i in range(17)], fontsize=8)
axes[2].set_xlabel('L/R Agreement Rate')
axes[2].set_title('Per-Item Left/Right Agreement')
axes[2].axvline(0.5, color='red', ls='--', alpha=0.3)
axes[2].set_xlim(0, 1)

plt.tight_layout()
plt.show()

## 5. Track 2 — Gait Subtype Distribution

In [ ]:
# === Cell 11: Track 2 — Gait Subtype Distribution ===
CLASS_NAMES = ['type1', 'type2', 'type3', 'type4', 'WNL']
CLASS_DISPLAY = {
    'type1': 'Type1: True Equinus',
    'type2': 'Type2: Jump Gait',
    'type3': 'Type3: Apparent Equinus',
    'type4': 'Type4: Crouch Gait',
    'WNL': 'WNL: Within Normal Limits',
}

left_types = [p['left']['gait_subtype'] for p in track2_labels]
right_types = [p['right']['gait_subtype'] for p in track2_labels]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 11a: Left limb
left_counts = Counter(left_types)
bars = axes[0].bar([CLASS_DISPLAY[c] for c in CLASS_NAMES],
                   [left_counts.get(c, 0) for c in CLASS_NAMES],
                   color=['#2196F3', '#FF9800', '#4CAF50', '#F44336', '#9E9E9E'])
axes[0].set_title(f'Left Limb (n={len(left_types)})')
axes[0].set_ylabel('Count')
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.1, str(int(h)), ha='center', fontsize=9)
axes[0].tick_params(axis='x', rotation=30)

# 11b: Right limb
right_counts = Counter(right_types)
bars = axes[1].bar([CLASS_DISPLAY[c] for c in CLASS_NAMES],
                   [right_counts.get(c, 0) for c in CLASS_NAMES],
                   color=['#2196F3', '#FF9800', '#4CAF50', '#F44336', '#9E9E9E'])
axes[1].set_title(f'Right Limb (n={len(right_types)})')
for bar in bars:
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, h + 0.1, str(int(h)), ha='center', fontsize=9)
axes[1].tick_params(axis='x', rotation=30)

# 11c: L/R agreement
all_combined = Counter(left_types) + Counter(right_types)
agree = sum(l == r for l, r in zip(left_types, right_types))
axes[2].bar([CLASS_DISPLAY[c] for c in CLASS_NAMES],
            [all_combined.get(c, 0) for c in CLASS_NAMES],
            color=['#2196F3', '#FF9800', '#4CAF50', '#F44336', '#9E9E9E'])
axes[2].set_title(f'Combined L+R (n={len(left_types)+len(right_types)})\nL/R Agreement: {agree}/{len(left_types)} ({agree/len(left_types):.0%})')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print(f'\nWARNING: type4 ({all_combined.get("type4",0)} samples) and WNL ({all_combined.get("WNL",0)} samples) are extremely rare!')
print(f'Track 2 only has {len(track2_labels)} patients — very small dataset.')

## 6. Keypoint sequence visualisation

Load one video for one patient and draw the skeleton at multiple frames.

In [ ]:
# === Cell 12: Load Video Keypoints Helper ===
def load_video_keypoints(video_path, keypoint_indices=None):
    """Load all frames from a video directory, return keypoints array.

    Args:
        video_path: path to video pose directory
        keypoint_indices: list of keypoint indices to extract (default: all 133)

    Returns:
        keypoints: np.array of shape (n_frames, n_keypoints, 2)
        scores: np.array of shape (n_frames, n_keypoints)
        frame_names: list of frame file names
    """
    frame_files = sorted([f for f in os.listdir(video_path) if f.endswith('.json')])
    keypoints_list = []
    scores_list = []

    for fname in frame_files:
        with open(os.path.join(video_path, fname)) as f:
            data = json.load(f)
        if not data['instance_info']:
            continue
        kps = np.array(data['instance_info'][0]['keypoints'])
        scs = np.array(data['instance_info'][0]['keypoint_scores'])

        if keypoint_indices is not None:
            kps = kps[keypoint_indices]
            scs = scs[keypoint_indices]

        keypoints_list.append(kps)
        scores_list.append(scs)

    return np.array(keypoints_list), np.array(scores_list), frame_files


def plot_skeleton(ax, kps, edges, scores=None, title=''):
    """Plot a single skeleton frame.

    Args:
        ax: matplotlib axes
        kps: (n_keypoints, 2) array
        edges: list of (i, j) tuples
        scores: optional (n_keypoints,) confidence scores
        title: plot title
    """
    ax.set_xlim(kps[:, 0].min() - 50, kps[:, 0].max() + 50)
    ax.set_ylim(kps[:, 1].max() + 50, kps[:, 1].min() - 50)  # Flip Y (image coords)

    # Draw edges
    for i, j in edges:
        if i < len(kps) and j < len(kps):
            ax.plot([kps[i, 0], kps[j, 0]], [kps[i, 1], kps[j, 1]],
                    'b-', linewidth=1, alpha=0.5)

    # Draw joints
    if scores is not None:
        colors = ['red' if s < 0.5 else 'orange' if s < 0.8 else 'green' for s in scores]
    else:
        colors = 'green'
    ax.scatter(kps[:, 0], kps[:, 1], c=colors, s=15, zorder=5)

    ax.set_title(title, fontsize=8)
    ax.set_aspect('equal')
    ax.axis('off')

print('Helpers loaded.')

In [ ]:
# === Cell 13: Visualize Skeleton — 4 Views of One Patient ===
# Pick a patient that has all 4 views
sample_pid = 1
sample_pid_str = f'{sample_pid:04d}'
sample_pid_dir = os.path.join(DATASET_DIR, sample_pid_str)
sample_videos = sorted([v for v in os.listdir(sample_pid_dir)
                        if os.path.isdir(os.path.join(sample_pid_dir, v))])

fig, axes = plt.subplots(1, len(sample_videos), figsize=(4 * len(sample_videos), 5))
if len(sample_videos) == 1:
    axes = [axes]

for ax, vname in zip(axes, sample_videos):
    vpath = os.path.join(sample_pid_dir, vname)
    kps, scs, _ = load_video_keypoints(vpath, keypoint_indices=GAIT_IDX)
    # Show middle frame
    mid = len(kps) // 2
    view_label = vname.split('_')[1]
    plot_skeleton(ax, kps[mid], SKELETON_EDGES, scores=scs[mid],
                  title=f'P{sample_pid} — {view_label}\n(frame {mid}/{len(kps)})')

fig.suptitle(f'Patient {sample_pid} — Body+Feet Keypoints (23 joints)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# === Cell 14: Skeleton Sequence Animation (single video) ===
# Pick a sagittal view (left/right) for best gait visualization
sample_row = df_stats[(df_stats['patient_id'] == sample_pid) &
                      (df_stats['view'] == 'right')].iloc[0]
kps, scs, fnames = load_video_keypoints(sample_row['video_path'], keypoint_indices=GAIT_IDX)

fig, ax = plt.subplots(figsize=(6, 6))

def animate(frame_idx):
    ax.clear()
    plot_skeleton(ax, kps[frame_idx], SKELETON_EDGES, scores=scs[frame_idx],
                  title=f'P{sample_pid} right view — frame {frame_idx}/{len(kps)}')
    # Fix axis limits across all frames for smooth animation
    ax.set_xlim(kps[:, :, 0].min() - 30, kps[:, :, 0].max() + 30)
    ax.set_ylim(kps[:, :, 1].max() + 30, kps[:, :, 1].min() - 30)

anim = FuncAnimation(fig, animate, frames=len(kps), interval=50)
plt.close()
HTML(anim.to_jshtml())

## 7. Gait-cycle detection prototype

Use the heel-to-midhip distance to detect heel strike (HS) and toe-off (TO).
- Sagittal view (left/right): use the **horizontal** (x) heel-midhip distance
- Coronal view (forward/backward): use the **vertical** (y) heel position

Once gait events are detected we can:
1. Split each video into individual gait cycles → multiplies training samples
2. Compute joint angles at key gait events → EVGS feature engineering

In [ ]:
# === Cell 15: Gait Cycle Detection Prototype ===
# Keypoint index mapping (within GAIT_IDX = 0-22)
KP = {name: i for i, name in enumerate(GAIT_KP_NAMES)}

def compute_midhip(kps_frame):
    """Compute midhip point from left_hip and right_hip."""
    return (kps_frame[KP['left_hip']] + kps_frame[KP['right_hip']]) / 2

def detect_gait_events_sagittal(kps, side='left'):
    """Detect heel strikes using heel-midhip horizontal distance in sagittal view.

    In sagittal view, heel strike = local maximum of heel-midhip x-distance.
    Toe-off = local minimum.

    Args:
        kps: (n_frames, 23, 2) keypoints
        side: 'left' or 'right'

    Returns:
        heel_strikes: list of frame indices
        toe_offs: list of frame indices
        signal: the heel-midhip distance signal
    """
    heel_key = f'{side}_heel'
    heel_idx = KP[heel_key]

    signal = []
    for t in range(len(kps)):
        midhip = compute_midhip(kps[t])
        heel = kps[t, heel_idx]
        # Horizontal distance (x-axis in sagittal view)
        dist = heel[0] - midhip[0]
        signal.append(dist)

    signal = np.array(signal)

    # Smooth with moving average
    kernel_size = 5
    if len(signal) > kernel_size:
        kernel = np.ones(kernel_size) / kernel_size
        signal_smooth = np.convolve(signal, kernel, mode='same')
    else:
        signal_smooth = signal

    # Find peaks (heel strikes) and valleys (toe-offs)
    from scipy.signal import find_peaks
    # Heel strike = peaks (foot is forward relative to hip)
    hs_indices, _ = find_peaks(signal_smooth, distance=15, prominence=5)
    # Toe-off = peaks of negated signal (foot is behind hip)
    to_indices, _ = find_peaks(-signal_smooth, distance=15, prominence=5)

    return hs_indices, to_indices, signal, signal_smooth

# Test on a sagittal video
sample_row = df_stats[(df_stats['patient_id'] == sample_pid) &
                      (df_stats['view'] == 'right')].iloc[0]
kps, _, _ = load_video_keypoints(sample_row['video_path'], keypoint_indices=GAIT_IDX)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

for ax, side, color in zip(axes, ['left', 'right'], ['steelblue', 'coral']):
    hs, to, signal, signal_smooth = detect_gait_events_sagittal(kps, side=side)

    ax.plot(signal, alpha=0.3, color=color, label='Raw signal')
    ax.plot(signal_smooth, color=color, linewidth=2, label='Smoothed')
    ax.scatter(hs, signal_smooth[hs], color='green', s=80, zorder=5,
               marker='v', label=f'Heel Strike (n={len(hs)})')
    ax.scatter(to, signal_smooth[to], color='red', s=80, zorder=5,
               marker='^', label=f'Toe Off (n={len(to)})')
    ax.set_ylabel('Heel-MidHip X Distance')
    ax.set_title(f'{side.capitalize()} Foot — Gait Events')
    ax.legend(fontsize=8)
    ax.axhline(0, color='gray', ls=':', alpha=0.3)

    # Highlight gait cycles
    for i in range(len(hs) - 1):
        ax.axvspan(hs[i], hs[i+1], alpha=0.05, color='green')

axes[1].set_xlabel('Frame')
fig.suptitle(f'Patient {sample_pid} — Sagittal View Gait Cycle Detection', y=1.01)
plt.tight_layout()
plt.show()

# Count total gait cycles
n_cycles_left = max(0, len(hs) - 1)
print(f'Detected gait cycles: Left={n_cycles_left}')

## 8. Joint-angle computation prototype

Compute the core joint angles used for gait analysis:
- **Knee angle**: hip-knee-ankle
- **Ankle angle**: knee-ankle-toe
- **Hip angle**: shoulder-hip-knee
- **Trunk angle**: shoulder midpoint vs hip midpoint vs vertical

In [ ]:
# === Cell 16: Joint Angle Computation ===
def compute_angle(p1, p2, p3):
    """Compute angle at p2 formed by p1-p2-p3 (in degrees).

    Args:
        p1, p2, p3: each is (2,) array [x, y]

    Returns:
        angle in degrees (0-180)
    """
    v1 = p1 - p2
    v2 = p3 - p2
    cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8)
    cos_angle = np.clip(cos_angle, -1, 1)
    return np.degrees(np.arccos(cos_angle))


def compute_joint_angles(kps_sequence, side='left'):
    """Compute key joint angles for all frames.

    Args:
        kps_sequence: (n_frames, 23, 2)
        side: 'left' or 'right'

    Returns:
        dict of angle_name -> np.array of shape (n_frames,)
    """
    s = side
    angles = defaultdict(list)

    for t in range(len(kps_sequence)):
        kp = kps_sequence[t]

        # Knee angle: hip-knee-ankle
        knee_angle = compute_angle(
            kp[KP[f'{s}_hip']], kp[KP[f'{s}_knee']], kp[KP[f'{s}_ankle']])
        angles['knee'].append(knee_angle)

        # Ankle angle: knee-ankle-heel (or big_toe)
        ankle_angle = compute_angle(
            kp[KP[f'{s}_knee']], kp[KP[f'{s}_ankle']], kp[KP[f'{s}_big_toe']])
        angles['ankle'].append(ankle_angle)

        # Hip angle: shoulder-hip-knee
        hip_angle = compute_angle(
            kp[KP[f'{s}_shoulder']], kp[KP[f'{s}_hip']], kp[KP[f'{s}_knee']])
        angles['hip'].append(hip_angle)

        # Trunk angle: midshoulder-midhip vs vertical
        midshoulder = (kp[KP['left_shoulder']] + kp[KP['right_shoulder']]) / 2
        midhip = compute_midhip(kp)
        trunk_vec = midshoulder - midhip
        vertical = np.array([0, -1])  # Up in image coords
        cos_a = np.dot(trunk_vec, vertical) / (np.linalg.norm(trunk_vec) + 1e-8)
        cos_a = np.clip(cos_a, -1, 1)
        angles['trunk'].append(np.degrees(np.arccos(cos_a)))

    return {k: np.array(v) for k, v in angles.items()}


# Compute for the same sagittal video
joint_angles_L = compute_joint_angles(kps, side='left')
joint_angles_R = compute_joint_angles(kps, side='right')

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

angle_names = ['knee', 'ankle', 'hip', 'trunk']
y_labels = ['Knee Angle (deg)', 'Ankle Angle (deg)', 'Hip Angle (deg)', 'Trunk Tilt (deg)']

for ax, name, ylabel in zip(axes, angle_names, y_labels):
    ax.plot(joint_angles_L[name], color='steelblue', label='Left', linewidth=1.5)
    ax.plot(joint_angles_R[name], color='coral', label='Right', linewidth=1.5)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

    # Mark heel strikes
    hs_L, _, _, _ = detect_gait_events_sagittal(kps, side='left')
    for h in hs_L:
        ax.axvline(h, color='steelblue', ls=':', alpha=0.3)
    hs_R, _, _, _ = detect_gait_events_sagittal(kps, side='right')
    for h in hs_R:
        ax.axvline(h, color='coral', ls=':', alpha=0.3)

axes[-1].set_xlabel('Frame')
fig.suptitle(f'Patient {sample_pid} — Joint Angles over Time (sagittal view)', y=1.01)
plt.tight_layout()
plt.show()

## 9. Multi-patient comparison - gait patterns across EVGS severity

Compare joint-angle patterns of patients with high (severe) vs low (mild) EVGS Total scores.

In [ ]:
# === Cell 17: Compare Joint Angles by EVGS Severity ===
# Sort patients by total EVGS score
patient_totals = []
for p in track1_labels:
    avg_total = (p['left']['Total'] + p['right']['Total']) / 2
    patient_totals.append((p['patient_id'], avg_total))
patient_totals.sort(key=lambda x: x[1])

# Pick 3 low-severity and 3 high-severity patients
low_pids = [pt[0] for pt in patient_totals[:3]]
high_pids = [pt[0] for pt in patient_totals[-3:]]
print(f'Low severity patients: {[(pid, tot) for pid, tot in patient_totals[:3]]}')
print(f'High severity patients: {[(pid, tot) for pid, tot in patient_totals[-3:]]}')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for row_idx, (pids, label, color_base) in enumerate([
    (low_pids, 'Low Severity', 'Blues'),
    (high_pids, 'High Severity', 'Reds')
]):
    for pid in pids:
        pid_str = f'{pid:04d}'
        pid_dir = os.path.join(DATASET_DIR, pid_str)
        if not os.path.exists(pid_dir):
            continue

        # Find a sagittal video (left or right view)
        videos = [v for v in os.listdir(pid_dir)
                  if os.path.isdir(os.path.join(pid_dir, v)) and
                  ('left' in v or 'right' in v)]
        if not videos:
            continue

        vpath = os.path.join(pid_dir, videos[0])
        kps_p, _, _ = load_video_keypoints(vpath, keypoint_indices=GAIT_IDX)
        if len(kps_p) < 10:
            continue

        angles_L = compute_joint_angles(kps_p, side='left')

        # Normalize time to 0-100% for comparison
        t_norm = np.linspace(0, 100, len(kps_p))

        total = next(p for p in track1_labels if p['patient_id'] == pid)
        total_score = total['left']['Total']

        axes[row_idx, 0].plot(t_norm, angles_L['knee'], alpha=0.7,
                              label=f'P{pid} (T={total_score})')
        axes[row_idx, 1].plot(t_norm, angles_L['ankle'], alpha=0.7,
                              label=f'P{pid} (T={total_score})')

    axes[row_idx, 0].set_title(f'{label} — Knee Angle')
    axes[row_idx, 0].set_ylabel('Angle (deg)')
    axes[row_idx, 0].legend(fontsize=7)
    axes[row_idx, 0].grid(alpha=0.2)

    axes[row_idx, 1].set_title(f'{label} — Ankle Angle')
    axes[row_idx, 1].legend(fontsize=7)
    axes[row_idx, 1].grid(alpha=0.2)

axes[1, 0].set_xlabel('Time (%)')
axes[1, 1].set_xlabel('Time (%)')
plt.suptitle('Joint Angles: Low vs High EVGS Severity', y=1.01)
plt.tight_layout()
plt.show()

## 10. Track 2 - joint-angle comparison across gait subtypes

Compare knee / ankle angle patterns across the four CP gait patterns plus WNL.

In [ ]:
# === Cell 18: Track 2 — Joint Angle Patterns by Gait Subtype ===
# Group track2 patients by left gait subtype
subtype_patients = defaultdict(list)
for p in track2_labels:
    subtype_patients[p['left']['gait_subtype']].append(p['patient_id'])

subtype_colors = {
    'type1': '#2196F3', 'type2': '#FF9800', 'type3': '#4CAF50',
    'type4': '#F44336', 'WNL': '#9E9E9E'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
angle_types = [('knee', 'Knee Angle'), ('ankle', 'Ankle Angle')]

for ax, (angle_name, angle_label) in zip(axes, angle_types):
    for subtype in CLASS_NAMES:
        pids = subtype_patients.get(subtype, [])
        if not pids:
            continue

        for pid in pids[:3]:  # Max 3 patients per subtype
            pid_str = f'{pid:04d}'
            pid_dir = os.path.join(DATASET_DIR, pid_str)
            if not os.path.exists(pid_dir):
                continue

            videos = [v for v in os.listdir(pid_dir)
                      if os.path.isdir(os.path.join(pid_dir, v)) and
                      ('left' in v or 'right' in v)]
            if not videos:
                continue

            vpath = os.path.join(pid_dir, videos[0])
            kps_p, _, _ = load_video_keypoints(vpath, keypoint_indices=GAIT_IDX)
            if len(kps_p) < 10:
                continue

            angles = compute_joint_angles(kps_p, side='left')
            t_norm = np.linspace(0, 100, len(kps_p))

            ax.plot(t_norm, angles[angle_name], color=subtype_colors[subtype],
                    alpha=0.5, linewidth=1)

    ax.set_xlabel('Time (%)')
    ax.set_ylabel(f'{angle_label} (deg)')
    ax.set_title(angle_label)
    ax.grid(alpha=0.2)

# Create legend
legend_elements = [mpatches.Patch(facecolor=subtype_colors[s], label=CLASS_DISPLAY[s])
                   for s in CLASS_NAMES if s in subtype_patients]
fig.legend(handles=legend_elements, loc='lower center', ncol=len(legend_elements),
           fontsize=8, bbox_to_anchor=(0.5, -0.05))
plt.suptitle('Track 2: Joint Angles by CP Gait Subtype', y=1.01)
plt.tight_layout()
plt.show()

## 11. Keypoint trajectories - per-joint motion over time

Inspect how individual keypoints move in x / y over time to confirm data quality.

In [ ]:
# === Cell 19: Keypoint Trajectory Visualization ===
# Use the same sagittal video loaded earlier (kps from Cell 15)
# kps shape: (n_frames, 23, 2)

# Plot lower-body keypoints x/y trajectories
lower_body_kps = ['left_hip', 'left_knee', 'left_ankle', 'left_heel', 'left_big_toe',
                  'right_hip', 'right_knee', 'right_ankle', 'right_heel', 'right_big_toe']

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

for kp_name in lower_body_kps:
    idx = KP[kp_name]
    color = 'steelblue' if 'left' in kp_name else 'coral'
    ls = '-' if 'hip' in kp_name or 'knee' in kp_name else '--'
    alpha = 0.8 if 'ankle' in kp_name or 'heel' in kp_name else 0.5

    # X trajectory
    axes[0].plot(kps[:, idx, 0], color=color, ls=ls, alpha=alpha,
                 linewidth=1, label=kp_name)
    # Y trajectory
    axes[1].plot(kps[:, idx, 1], color=color, ls=ls, alpha=alpha,
                 linewidth=1, label=kp_name)

axes[0].set_ylabel('X position (px)')
axes[0].set_title('Lower Body Keypoint X Trajectories (sagittal view)')
axes[0].legend(fontsize=6, ncol=2, loc='upper right')
axes[0].grid(alpha=0.2)

axes[1].set_ylabel('Y position (px)')
axes[1].set_xlabel('Frame')
axes[1].set_title('Lower Body Keypoint Y Trajectories (sagittal view)')
axes[1].legend(fontsize=6, ncol=2, loc='upper right')
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

# Check for jumps / outliers in keypoint positions
print('Keypoint position statistics (body+feet):')
print(f'  X range: [{kps[:,:,0].min():.0f}, {kps[:,:,0].max():.0f}] px')
print(f'  Y range: [{kps[:,:,1].min():.0f}, {kps[:,:,1].max():.0f}] px')

# Frame-to-frame displacement check
diffs = np.diff(kps, axis=0)  # (n_frames-1, 23, 2)
max_displacement = np.sqrt((diffs**2).sum(axis=-1)).max(axis=0)  # per keypoint
print(f'\nMax frame-to-frame displacement per keypoint (px):')
for i, name in enumerate(GAIT_KP_NAMES):
    if max_displacement[i] > 30:
        print(f'  WARNING {name}: {max_displacement[i]:.1f} px')
    else:
        print(f'  {name}: {max_displacement[i]:.1f} px')

## 12. Global summary statistics

Scan all patients' sagittal views, count gait cycles, and estimate how much gait-cycle splitting would multiply the training-sample count.

In [ ]:
# === Cell 20: Global Gait Cycle Count ===
# Scan all sagittal videos (left/right views) for gait cycles
from scipy.signal import find_peaks

total_cycles = 0
cycle_counts = []

sagittal_videos = df_stats[df_stats['view'].isin(['left', 'right'])]
print(f'Scanning {len(sagittal_videos)} sagittal videos for gait cycles...')

for _, row in sagittal_videos.iterrows():
    try:
        kps_v, _, _ = load_video_keypoints(row['video_path'], keypoint_indices=GAIT_IDX)
        if len(kps_v) < 20:
            cycle_counts.append(0)
            continue

        # Try both sides
        max_cycles = 0
        for side in ['left', 'right']:
            hs, _, _, _ = detect_gait_events_sagittal(kps_v, side=side)
            n_cycles = max(0, len(hs) - 1)
            max_cycles = max(max_cycles, n_cycles)

        cycle_counts.append(max_cycles)
        total_cycles += max_cycles
    except Exception:
        cycle_counts.append(0)

print(f'\nTotal gait cycles detected: {total_cycles}')
print(f'Cycles per video: mean={np.mean(cycle_counts):.1f}, '
      f'median={np.median(cycle_counts):.0f}, '
      f'min={np.min(cycle_counts)}, max={np.max(cycle_counts)}')
print(f'\nSample multiplication: {len(sagittal_videos)} videos → ~{total_cycles} gait cycles')
print(f'(+{total_cycles - len(sagittal_videos)} additional training samples from cycle splitting)')

## 13. EDA notes

Reference questions guiding the modelling choices in `notebooks/01_preprocess` and `notebooks/02_train_pipeline`:

### Data structure
- 133 COCO-WholeBody keypoints; gait analysis uses only body + feet (23 keypoints).
- Each patient has multiple sessions and a few view directions per session.

### Label distribution
- Track 1: 17 EVGS items per side, with class imbalance varying by item.
- Track 2: 22 training patients across 5 gait subtypes (type1-4, WNL).

### Feature-engineering choices made downstream
- Drop face / hand keypoints (lower confidence; not gait-relevant).
- Use gait-cycle splitting to multiply per-patient training samples.
- View routing: sagittal items use sagittal-view features; coronal items use coronal-view features.
